# 06 — Evaluation: ROC, Reliability Diagram & ECE (Classical Pipeline)

This notebook evaluates the **calibration and discrimination** of the GMM-based Bayesian classifier on the held-out test set.

| Metric | What it measures |
|--------|------------------|
| **AUC-ROC** | Discrimination — how well the model ranks positives above negatives, independent of threshold |
| **Reliability diagram** | Calibration — whether predicted probabilities match empirical frequencies |
| **ECE** | Scalar summary of calibration error (expected |accuracy − confidence| weighted by bin size) |

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path(os.getcwd())
for _root in [_cwd, *_cwd.parents]:
    if (_root / "skin_lesion" / "src" / "config.py").exists():
        sys.path.insert(0, str(_root / "skin_lesion" / "src"))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_curve, roc_auc_score

from config import PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, COST_FN, COST_FP

## 1 — Load test set and recompute posteriors

Posteriors are recomputed here with the same `compute_posterior` logic used in notebook 05,
ensuring this notebook is self-contained and reproducible independently.

In [ ]:
data = np.load(PROCESSED_DIR / "splits.npz")
X_test, y_test = data["X_test"], data["y_test"]

gmm_benign   = joblib.load(PROCESSED_DIR / "gmm_benign.pkl")
gmm_melanoma = joblib.load(PROCESSED_DIR / "gmm_melanoma.pkl")

def compute_posterior(X, gmm_mel, gmm_ben):
    """P(omega_M | x) via sigmoid of log-likelihood ratio (numerically stable)."""
    log_ratio = np.clip(
        gmm_mel.score_samples(X) - gmm_ben.score_samples(X),
        -500, 500
    )
    return 1.0 / (1.0 + np.exp(-log_ratio))

probs = compute_posterior(X_test, gmm_melanoma, gmm_benign)

print(f"Test samples : {len(y_test)}")
print(f"Posterior    : mean={probs.mean():.4f}, std={probs.std():.4f}, "
      f"min={probs.min():.4f}, max={probs.max():.4f}")

## 2 — ROC curve

The ROC curve sweeps all possible classification thresholds and plots TPR (sensitivity) vs FPR (1 − specificity).  
AUC = 1.0 is perfect; AUC = 0.5 is random.  

We mark the two operating points from notebook 05:
- **MAP** (θ = 0.5): optimal under equal costs
- **Cost-sensitive** (θ = 1/6): optimal under λ(FN) = 5·λ(FP)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, probs)
auc = roc_auc_score(y_test, probs)

THRESH_MAP  = 0.5
THRESH_COST = COST_FP / (COST_FP + COST_FN)  # 1/6

def point_on_roc(threshold, fpr_arr, tpr_arr, thresh_arr):
    """Find the (FPR, TPR) operating point closest to a given threshold."""
    idx = np.argmin(np.abs(thresh_arr - threshold))
    return fpr_arr[idx], tpr_arr[idx]

fpr_map,  tpr_map  = point_on_roc(THRESH_MAP,  fpr, tpr, thresholds)
fpr_cost, tpr_cost = point_on_roc(THRESH_COST, fpr, tpr, thresholds)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, color="steelblue", label=f"GMM classifier (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="Random (AUC = 0.5)")
ax.scatter(fpr_map,  tpr_map,  s=80, zorder=5, color="tomato",
           label=f"MAP  θ=0.5  (FPR={fpr_map:.3f}, TPR={tpr_map:.3f})")
ax.scatter(fpr_cost, tpr_cost, s=80, zorder=5, color="darkorange", marker="D",
           label=f"Cost  θ=1/6  (FPR={fpr_cost:.3f}, TPR={tpr_cost:.3f})")
ax.set_xlabel("False Positive Rate (1 − Specificity)")
ax.set_ylabel("True Positive Rate (Sensitivity)")
ax.set_title("ROC Curve — Classical GMM Pipeline")
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, linestyle=":")
plt.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"AUC = {auc:.4f}")
print(f"Figure saved to: {FIGURES_DIR / 'roc_curve.png'}")

## 3 — Reliability diagram

A well-calibrated classifier has P(Y=1 | p̂ ≈ p) = p for all p.  
The reliability diagram checks this empirically: samples are grouped into equal-width confidence bins,  
and for each bin we plot the **mean predicted probability** (confidence) against the **fraction of positives** (accuracy).  
Bars above the diagonal → underconfidence; bars below → overconfidence.

In [ ]:
def reliability_diagram(probs, labels, n_bins=15):
    """
    Compute calibration statistics.

    Returns
    -------
    bin_centers   : (n_bins,) midpoints of each bin
    bin_conf      : (n_bins,) mean predicted probability per bin (NaN if empty)
    bin_acc       : (n_bins,) fraction of positives per bin (NaN if empty)
    bin_counts    : (n_bins,) number of samples per bin
    """
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_conf    = np.full(n_bins, np.nan)
    bin_acc     = np.full(n_bins, np.nan)
    bin_counts  = np.zeros(n_bins, dtype=int)

    for m in range(n_bins):
        mask = (probs >= bins[m]) & (probs < bins[m + 1])
        # include right edge in last bin
        if m == n_bins - 1:
            mask |= (probs == bins[m + 1])
        if mask.sum() > 0:
            bin_conf[m]   = probs[mask].mean()
            bin_acc[m]    = labels[mask].mean()
            bin_counts[m] = mask.sum()

    return bin_centers, bin_conf, bin_acc, bin_counts


N_BINS = 15
bin_centers, bin_conf, bin_acc, bin_counts = reliability_diagram(probs, y_test, N_BINS)

fig, ax = plt.subplots(figsize=(6, 5))
# bar width slightly narrower than bin width so gaps are visible
width = 1.0 / N_BINS * 0.85
valid = ~np.isnan(bin_conf)
ax.bar(bin_centers[valid], bin_acc[valid], width=width,
       color="steelblue", alpha=0.7, label="Fraction of positives")
ax.plot([0, 1], [0, 1], "--", color="tomato", lw=1.5, label="Perfect calibration")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean predicted probability (confidence)")
ax.set_ylabel("Fraction of positives (accuracy)")
ax.set_title("Reliability Diagram — Classical GMM Pipeline")
ax.legend(fontsize=9)
ax.grid(True, linestyle=":")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "reliability_diagram.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to: {FIGURES_DIR / 'reliability_diagram.png'}")

## 4 — Expected Calibration Error (ECE)

$$\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n} \left| \text{acc}(B_m) - \text{conf}(B_m) \right|$$

where B_m is the set of samples in bin m, n is the total number of samples,  
acc(B_m) is the fraction of positives in B_m, and conf(B_m) is the mean predicted probability in B_m.  
ECE = 0 is perfect calibration; ECE = 1 is maximally miscalibrated.

In [ ]:
def expected_calibration_error(probs, labels, n_bins=15):
    """ECE as defined in the course slides."""
    _, bin_conf, bin_acc, bin_counts = reliability_diagram(probs, labels, n_bins)
    n = len(labels)
    valid = ~np.isnan(bin_conf)
    ece = np.sum(
        (bin_counts[valid] / n) * np.abs(bin_acc[valid] - bin_conf[valid])
    )
    return float(ece)


ece = expected_calibration_error(probs, y_test, n_bins=N_BINS)
print(f"ECE (classical GMM, test set, {N_BINS} bins) = {ece:.4f}")

## 5 — Calibration summary table

In [ ]:
summary = pd.DataFrame([{
    "model":  "classical_gmm",
    "auc":    round(auc, 4),
    "ece":    round(ece, 4),
    "n_bins": N_BINS,
}])
print(summary.to_string(index=False))

TABLES_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(TABLES_DIR / "calibration_summary.csv", index=False)
print(f"\nSaved to: {TABLES_DIR / 'calibration_summary.csv'}")

## 6 — Calibration interpretation

### What to expect from a GMM-based classifier

GMMs model class-conditional densities, not the posterior directly.  
Calibration quality therefore depends on how well the Gaussian mixture assumptions match the true feature distribution.

### Reading the reliability diagram

**Bars above the diagonal** (bin accuracy > bin confidence): the model is **underconfident** — it assigns moderate probabilities even when the label is almost certainly melanoma. This is common when the class-conditional distributions heavily overlap in feature space.

**Bars below the diagonal** (bin accuracy < bin confidence): the model is **overconfident** — it assigns very high or very low probabilities more often than the empirical frequency justifies. This happens when the GMM places most probability mass far from the decision boundary, producing posteriors near 0 or 1.

**U-shaped or bi-modal distribution of posteriors**: if the model outputs mostly values near 0 or 1 with few intermediate values, the reliability diagram will have populated bars only at the extremes. This is a sign that the GMM has learned well-separated densities — potentially good for AUC, but it may indicate overconfidence if the extreme bins have lower-than-expected accuracy.

### Implications for the cost-sensitive threshold

If the GMM is overconfident (posteriors cluster near 0 or 1), shifting the threshold from 0.5 to 1/6 may have little practical effect: samples already assigned posterior < 1/6 remain benign, and those > 0.5 remain melanoma — only the narrow band [1/6, 0.5] is affected.  
Poor calibration therefore limits the benefit of the asymmetric cost rule, which is one motivation for comparing against the modern (EfficientNet) pipeline, where calibration can be improved with temperature scaling or label smoothing.